In [1]:

# Import required libraries

import pandas as pd
import numpy as np
import glob

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("All required libraries imported successfully!")

# Locate and load dataset
matches = glob.glob("/content/Food_Delivery_Customer_Churn*.csv") + glob.glob("Food_Delivery_Customer_Churn*.csv")

if not matches:
    raise FileNotFoundError(
        "Food Delivery CSV not found. Upload it using the Files panel."
    )

csv_file = matches[0]
df = pd.read_csv(csv_file)

print("\nDataset loaded successfully!")
print("File:", csv_file)


All required libraries imported successfully!

Dataset loaded successfully!
File: Food_Delivery_Customer_Churn.csv


In [2]:

print("===== FOOD DELIVERY CUSTOMER CHURN DATASET =====")

print("\nFirst 5 Records:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nChurn Distribution:")
print(df["Churn"].value_counts())


===== FOOD DELIVERY CUSTOMER CHURN DATASET =====

First 5 Records:
   Age         City Subscription_Type  Orders_Per_Month  Average_Order_Value  \
0   56        Thane              Free                 2                  151   
1   46       Mumbai              Free                 8                  738   
2   32         Pune              Free                13                  793   
3   60  Navi Mumbai           Premium                13                  239   
4   25  Navi Mumbai              Free                 1                  499   

   Average_Delivery_Time  Discount_Usage Favorite_Category   Device  \
0                     36              39            Indian  Android   
1                     42              13           Chinese      Web   
2                     21              34            Indian      iOS   
3                     49              47           Mexican  Android   
4                     69              83           Burgers      iOS   

   Tenure_Months     Chur

In [3]:

# Encode categorical attributes

categorical_columns = [
    "City",
    "Subscription_Type",
    "Favorite_Category",
    "Device"
]

encoders = {}

for column in categorical_columns:
    encoder = LabelEncoder()
    df[column] = encoder.fit_transform(df[column])
    encoders[column] = encoder

# Encode target variable
target_encoder = LabelEncoder()
df["Churn"] = target_encoder.fit_transform(df["Churn"])

print("===== ENCODING MAPPINGS =====")

for column, encoder in encoders.items():
    print(column, ":", dict(zip(
        encoder.classes_,
        encoder.transform(encoder.classes_)
    )))

print("\nChurn Encoding:", dict(zip(
    target_encoder.classes_,
    target_encoder.transform(target_encoder.classes_)
)))

# Separate features and target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("\n===== DATA PREPROCESSING COMPLETED =====")

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])
print("Number of Features:", X_train.shape[1])

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())



===== ENCODING MAPPINGS =====
City : {'Mumbai': np.int64(0), 'Navi Mumbai': np.int64(1), 'Pune': np.int64(2), 'Thane': np.int64(3)}
Subscription_Type : {'Free': np.int64(0), 'Premium': np.int64(1)}
Favorite_Category : {'Burgers': np.int64(0), 'Chinese': np.int64(1), 'Indian': np.int64(2), 'Mexican': np.int64(3), 'Pizza': np.int64(4)}
Device : {'Android': np.int64(0), 'Web': np.int64(1), 'iOS': np.int64(2)}

Churn Encoding: {'Churned': np.int64(0), 'Retained': np.int64(1)}

===== DATA PREPROCESSING COMPLETED =====
Training Samples: 400
Testing Samples: 100
Number of Features: 10

Training Target Distribution:
Churn
1    247
0    153
Name: count, dtype: int64

Testing Target Distribution:
Churn
1    62
0    38
Name: count, dtype: int64


In [4]:

# Inbuilt Decision Tree Classifier

decision_tree_inbuilt = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

# Train the model
decision_tree_inbuilt.fit(X_train, y_train)

# Predict test data
y_pred_dt_inbuilt = decision_tree_inbuilt.predict(X_test)

# Calculate accuracy
accuracy_dt_inbuilt = accuracy_score(
    y_test,
    y_pred_dt_inbuilt
)

print("Inbuilt Decision Tree trained successfully!")
print("Criterion: Gini")
print("Maximum Depth: 5")


Inbuilt Decision Tree trained successfully!
Criterion: Gini
Maximum Depth: 5


In [5]:

print("===== INBUILT DECISION TREE RESULTS =====")

print(
    "Accuracy:",
    round(accuracy_dt_inbuilt * 100, 2),
    "%"
)

print("\n===== CLASSIFICATION REPORT =====")

print(classification_report(
    y_test,
    y_pred_dt_inbuilt,
    labels=[0, 1],
    target_names=["Churned", "Retained"],
    zero_division=0
))

print("\n===== CONFUSION MATRIX =====")

print(confusion_matrix(
    y_test,
    y_pred_dt_inbuilt,
    labels=[0, 1]
))


===== INBUILT DECISION TREE RESULTS =====
Accuracy: 72.0 %

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

     Churned       0.68      0.50      0.58        38
    Retained       0.74      0.85      0.79        62

    accuracy                           0.72       100
   macro avg       0.71      0.68      0.68       100
weighted avg       0.71      0.72      0.71       100


===== CONFUSION MATRIX =====
[[19 19]
 [ 9 53]]


In [6]:

# User-Defined Decision Tree Function

def decision_tree_classifier(
    X_train, X_test, y_train, y_test
):

    # Create Decision Tree model
    model = DecisionTreeClassifier(
        criterion="entropy",
        max_depth=5,
        random_state=42
    )

    # Train model
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(
        y_test, predictions
    )

    return model, predictions, accuracy


# Call user-defined function
(
    decision_tree_user,
    y_pred_dt_user,
    accuracy_dt_user
) = decision_tree_classifier(
    X_train, X_test, y_train, y_test
)

print("User-defined Decision Tree executed successfully!")
print("Criterion: Entropy")
print("Maximum Depth: 5")


User-defined Decision Tree executed successfully!
Criterion: Entropy
Maximum Depth: 5


In [7]:

print("===== USER-DEFINED DECISION TREE RESULTS =====")

print(
    "Accuracy:",
    round(accuracy_dt_user * 100, 2),
    "%"
)

print("\n===== CLASSIFICATION REPORT =====")

print(classification_report(
    y_test,
    y_pred_dt_user,
    labels=[0, 1],
    target_names=["Churned", "Retained"],
    zero_division=0
))

print("\n===== CONFUSION MATRIX =====")

print(confusion_matrix(
    y_test,
    y_pred_dt_user,
    labels=[0, 1]
))


===== USER-DEFINED DECISION TREE RESULTS =====
Accuracy: 74.0 %

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

     Churned       0.75      0.47      0.58        38
    Retained       0.74      0.90      0.81        62

    accuracy                           0.74       100
   macro avg       0.74      0.69      0.70       100
weighted avg       0.74      0.74      0.72       100


===== CONFUSION MATRIX =====
[[18 20]
 [ 6 56]]


In [8]:

# Inbuilt Naive Bayes Classifier

naive_bayes_inbuilt = GaussianNB()

# Train the model
naive_bayes_inbuilt.fit(X_train, y_train)

# Predict test data
y_pred_nb_inbuilt = naive_bayes_inbuilt.predict(X_test)

# Calculate accuracy
accuracy_nb_inbuilt = accuracy_score(
    y_test,
    y_pred_nb_inbuilt
)

print("Inbuilt Naive Bayes trained successfully!")
print("Algorithm: Gaussian Naive Bayes")


Inbuilt Naive Bayes trained successfully!
Algorithm: Gaussian Naive Bayes


In [9]:

print("===== INBUILT NAIVE BAYES RESULTS =====")

print(
    "Accuracy:",
    round(accuracy_nb_inbuilt * 100, 2),
    "%"
)

print("\n===== CLASSIFICATION REPORT =====")

print(classification_report(
    y_test,
    y_pred_nb_inbuilt,
    labels=[0, 1],
    target_names=["Churned", "Retained"],
    zero_division=0
))

print("\n===== CONFUSION MATRIX =====")

print(confusion_matrix(
    y_test,
    y_pred_nb_inbuilt,
    labels=[0, 1]
))


===== INBUILT NAIVE BAYES RESULTS =====
Accuracy: 74.0 %

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

     Churned       0.69      0.58      0.63        38
    Retained       0.76      0.84      0.80        62

    accuracy                           0.74       100
   macro avg       0.73      0.71      0.71       100
weighted avg       0.74      0.74      0.73       100


===== CONFUSION MATRIX =====
[[22 16]
 [10 52]]


In [10]:

# User-Defined Naive Bayes Function

def naive_bayes_classifier(
    X_train, X_test, y_train, y_test
):

    # Create Naive Bayes model
    model = GaussianNB()

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(
        y_test, predictions
    )

    return model, predictions, accuracy


# Call user-defined function
(
    naive_bayes_user,
    y_pred_nb_user,
    accuracy_nb_user
) = naive_bayes_classifier(
    X_train, X_test, y_train, y_test
)

print("User-defined Naive Bayes executed successfully!")


User-defined Naive Bayes executed successfully!


In [11]:

print("===== USER-DEFINED NAIVE BAYES RESULTS =====")

print(
    "Accuracy:",
    round(accuracy_nb_user * 100, 2),
    "%"
)

print("\n===== CLASSIFICATION REPORT =====")

print(classification_report(
    y_test,
    y_pred_nb_user,
    labels=[0, 1],
    target_names=["Churned", "Retained"],
    zero_division=0
))

print("\n===== CONFUSION MATRIX =====")

print(confusion_matrix(
    y_test,
    y_pred_nb_user,
    labels=[0, 1]
))


===== USER-DEFINED NAIVE BAYES RESULTS =====
Accuracy: 74.0 %

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

     Churned       0.69      0.58      0.63        38
    Retained       0.76      0.84      0.80        62

    accuracy                           0.74       100
   macro avg       0.73      0.71      0.71       100
weighted avg       0.74      0.74      0.73       100


===== CONFUSION MATRIX =====
[[22 16]
 [10 52]]


In [12]:

# Compare all four classifiers

comparison_data = {
    "Classifier": [
        "Inbuilt Decision Tree",
        "User-Defined Decision Tree",
        "Inbuilt Naive Bayes",
        "User-Defined Naive Bayes"
    ],

    "Accuracy (%)": [
        accuracy_dt_inbuilt * 100,
        accuracy_dt_user * 100,
        accuracy_nb_inbuilt * 100,
        accuracy_nb_user * 100
    ]
}

comparison_df = pd.DataFrame(comparison_data)

# Identify the best classifier
best_index = comparison_df["Accuracy (%)"].idxmax()

best_classifier = comparison_df.loc[
    best_index, "Classifier"
]

best_accuracy = comparison_df.loc[
    best_index, "Accuracy (%)"
]

print("Classifier comparison completed successfully!")


Classifier comparison completed successfully!


In [13]:

print("===== CLASSIFIER PERFORMANCE COMPARISON =====")

print(comparison_df.round(2).to_string(index=False))

print("\n===== BEST CLASSIFIER =====")

print("Classifier:", best_classifier)

print("Accuracy:", round(best_accuracy, 2), "%")


===== CLASSIFIER PERFORMANCE COMPARISON =====
                Classifier  Accuracy (%)
     Inbuilt Decision Tree          72.0
User-Defined Decision Tree          74.0
       Inbuilt Naive Bayes          74.0
  User-Defined Naive Bayes          74.0

===== BEST CLASSIFIER =====
Classifier: User-Defined Decision Tree
Accuracy: 74.0 %


In [14]:
# Control run: inbuilt Decision Tree with the SAME criterion (Entropy) as the user-defined one
# This shows that the 72% vs 74% gap comes from Gini vs Entropy, not from the function wrapper.

dt_entropy_inbuilt = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

dt_entropy_inbuilt.fit(X_train, y_train)
y_pred_dt_entropy = dt_entropy_inbuilt.predict(X_test)

print("===== CONTROL: INBUILT DECISION TREE (ENTROPY) =====")
print("Accuracy:", round(accuracy_score(y_test, y_pred_dt_entropy) * 100, 2), "%")
print("Identical predictions to user-defined Decision Tree:",
      bool((y_pred_dt_entropy == y_pred_dt_user).all()))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt_entropy, labels=[0, 1]))


===== CONTROL: INBUILT DECISION TREE (ENTROPY) =====
Accuracy: 74.0 %
Identical predictions to user-defined Decision Tree: True

Confusion Matrix:
[[18 20]
 [ 6 56]]
